In [0]:
dbutils.widgets.text("run_date", "")
run_date = dbutils.widgets.get("run_date")
print(f"Running bronze for: {run_date or 'all data'}")

Running bronze for: all data


In [0]:
%python
from pyspark.sql.functions import current_timestamp, col

src_dir = "/Volumes/workspace/default/raw/"
chk     = "/Volumes/workspace/default/checkpoints/bronze_job_postings"

df = (spark.readStream
      .format("cloudFiles")                                      # this is Auto Loader
      .option("cloudFiles.format", "csv")
      .option("cloudFiles.schemaLocation", f"{chk}/schema")      # remembers the columns
      .option("pathGlobFilter", "neo4j_query_table_data_*.csv")  # only your export files
      .option("header", True)
      .option("multiLine", True)
      .option("escape", '"')
      .load(src_dir)
      .withColumn("_ingested_at", current_timestamp())
      .withColumn("_source_file", col("_metadata.file_path")))

(df.writeStream
   .option("checkpointLocation", chk)      # remembers which files are already loaded
   .trigger(availableNow=True)             # load all new files, then stop
   .toTable("workspace.default.bronze_job_postings")
   .awaitTermination())

In [0]:
%python
t = spark.table("workspace.default.bronze_job_postings")
print("Columns:", t.columns)
print("Rows:", t.count(), "| distinct job_id:", t.select("job_id").distinct().count())
print("Files loaded so far:")
t.groupBy("_source_file").count().show(truncate=False)

Columns: ['job_id', 'title', 'company', 'location', 'work_type', 'remote', 'url', 'description', '_rescued_data', '_ingested_at', '_source_file']
Rows: 837 | distinct job_id: 837
Files loaded so far:
+--------------------------------------------------------------------+-----+
|_source_file                                                        |count|
+--------------------------------------------------------------------+-----+
|/Volumes/workspace/default/raw/neo4j_query_table_data_2026-10-28.csv|804  |
|/Volumes/workspace/default/raw/neo4j_query_table_data_2026-10-27.csv|33   |
+--------------------------------------------------------------------+-----+



In [0]:
row_count = spark.table("workspace.default.bronze_job_postings").count()
dbutils.jobs.taskValues.set(key="bronze_rows", value=row_count)
print(f"Bronze rows: {row_count}")

Bronze rows: 837


## Bronze: job postings
Source: Neo4j export, 804 jobs, loaded as-is from /Volumes/workspace/default/raw/.
Issues found and fixed during ingestion:
- Export used an inner join and dropped 9 jobs with no company (795 rows). Fixed with OPTIONAL MATCH.
- Export used j.id, which doesn't exist, so job_id was null for every row. Fixed to j.job_id.
Known: 9 jobs have company = "null" text; cleaned in silver.